In [1]:
import os
import tifffile
import rasterio
import glob
import cv2
import numpy as np
import leafmap.leafmap as leafmap
import pickle

import matplotlib.pyplot as plt
from utils.raster_tools import Raster_profile

# Merge 10x10 file & Evaluate the Building density


## Merge file

In [2]:
import glob
import leafmap.leafmap as leafmap

google_path_searching = os.path.join("Sorted_Data", "*google.tif")
mask_path_searching   = os.path.join("Sorted_Data", "*masks.tif")

filename_google = glob.glob(google_path_searching)
filename_masks  = glob.glob(mask_path_searching)

m = leafmap.Map()    
for filename in filename_masks:
    slice_theos_filename_prev_test = filename
    m.add_raster(slice_theos_filename_prev_test, layer_name="Theos-prev") 

 

# Counting Building

In [3]:
import os
import numpy as np
import io
import contextlib
from utils.mask_tools import Mask_profile

mask_root = 'Sorted_Data'
total_num_obs = 0

for row in range(1, 11):
    for col in range(1, 11):
        tif_path = os.path.join(mask_root, f"{row}-{col}_masks.tif")
        tiff_path = os.path.join(mask_root, f"{row}-{col}_masks.tiff")

        if os.path.exists(tif_path):
            mask_path = tif_path
        elif os.path.exists(tiff_path):
            mask_path = tiff_path
        else:
            continue

        with contextlib.redirect_stdout(io.StringIO()):
            Mask_obj_top = Mask_profile(mask_path)

        object_id = np.unique(Mask_obj_top.mask)
        object_id = object_id[object_id > 0]
        num_obs = len(object_id)
        
        print(f"{os.path.basename(mask_path)} : {num_obs}")
        total_num_obs += num_obs

print(f"Total : {total_num_obs}")

1-1_masks.tif : 49
1-2_masks.tif : 78
1-3_masks.tif : 84
1-4_masks.tif : 163
1-5_masks.tif : 23
1-6_masks.tif : 57
1-7_masks.tif : 129
1-8_masks.tif : 28
1-9_masks.tif : 65
1-10_masks.tif : 83
2-1_masks.tif : 64
2-2_masks.tif : 93
2-3_masks.tif : 103
2-4_masks.tif : 90
2-5_masks.tif : 91
2-6_masks.tif : 88
2-7_masks.tif : 129
2-8_masks.tif : 222
2-9_masks.tif : 93
2-10_masks.tif : 150
3-1_masks.tif : 70
3-2_masks.tif : 132
3-3_masks.tif : 165
3-4_masks.tif : 106
3-5_masks.tif : 41
3-6_masks.tif : 65
3-7_masks.tif : 188
3-8_masks.tif : 276
3-9_masks.tif : 147
3-10_masks.tif : 128
4-1_masks.tif : 80
4-2_masks.tif : 147
4-3_masks.tif : 221
4-4_masks.tif : 123
4-5_masks.tif : 72
4-6_masks.tif : 51
4-7_masks.tif : 148
4-8_masks.tif : 289
4-9_masks.tif : 165
4-10_masks.tif : 177
5-1_masks.tif : 87
5-2_masks.tif : 109
5-3_masks.tif : 120
5-4_masks.tif : 174
5-5_masks.tif : 106
5-6_masks.tif : 41
5-7_masks.tif : 272
5-8_masks.tif : 274
5-9_masks.tif : 204
5-10_masks.tif : 220
6-1_masks.tif : 8

# Calculate Area from Latitute & Longtitude

In [4]:
import math
lon_start = 100.886455 ; lat_start = 12.942518
lon_end = 100.918329 ; lat_end = 12.910644

x = abs(lat_start - lat_end) * 110.574

lat_avg = (lat_start + lat_end) / 2

y = abs(lon_end - lon_start) * 111.320 * math.cos(math.radians(lat_avg)) # คำนวณความกว้างของพื้นที่ในหน่วยกิโลเมตร
area = min(x, y) ** 2 # ถ้าใช้ x*y จะได้พื้นที่จริง แต่ถ้าใช้ min(x,y)**2 จะได้พื้นที่ที่เป็นสี่เหลี่ยมจัตุรัสที่ครอบคลุมทั้งพื้นที่ (ซึ่งเหมาะกับการแบ่งเป็นตาราง)

#print(f"X = {x:.6f} km")
#print(f"Y = {y:.6f} km")
print(f"Area = {area:.6f} km^2")

Area = 11.959791 km^2


# Calculate Density

In [5]:
density = total_num_obs/area
print(f"Density = {density:.6f} obs/km^2")

Density = 1040.235532 obs/km^2


# Load from local

In [6]:
import os
import re

# กำหนด Path หลัก
root_path = r'E:/GISTDA/geo/GeoAnnotation/raw_data'

all_files_found = []

# Regex สำหรับเช็คชื่อโฟลเดอร์ที่เป็นตัวเลข (เช่น 1-5, 10-10)
folder_pattern = re.compile(r'^\d+-\d+$')

for root, dirs, files in os.walk(root_path):
    for filename in files:
        if filename in ['google.tif', 'theos.tif', 'masks.tif']:
            
            # แยกส่วนประกอบของ Path ทั้งหมดออกเป็น List
            path_parts = root.split(os.sep)
            
            # วนลูปถอยหลังจากโฟลเดอร์ปัจจุบันขึ้นไป เพื่อหาโฟลเดอร์ที่ชื่อตรงตาม pattern (ตัวเลข-ตัวเลข)
            prefix = "unknown"
            for part in reversed(path_parts):
                if folder_pattern.match(part):
                    prefix = part
                    break
            
            # สร้างชื่อใหม่
            new_display_name = f"{prefix}_{filename}"
            
            all_files_found.append({
                'new_name': new_display_name,
                'path': os.path.join(root, filename)
            })

# --- แสดงผลลัพธ์เพื่อตรวจสอบ ---
for file_info in all_files_found:
    print(f"Path: {file_info['path']}")
    print(f"Result -> {file_info['new_name']}")
    print("-" * 30)

Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-1\google.tif
Result -> 1-1_google.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-1\theos.tif
Result -> 1-1_theos.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-1\samgeo2mask\masks.tif
Result -> 1-1_masks.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-10\google.tif
Result -> 1-10_google.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-10\theos.tif
Result -> 1-10_theos.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-10\samgeo2mask\masks.tif
Result -> 1-10_masks.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-2\google.tif
Result -> 1-2_google.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-2\theos.tif
Result -> 1-2_theos.tif
------------------------------
Path: E:/GISTDA/geo/GeoAnnotation/raw_data\1-2\samgeo2mask\masks.tif

In [7]:

folder_pattern = re.compile(r'^\d+-\d+$')

# ใช้ Dictionary เก็บข้อมูลเป็นกลุ่มๆ ตาม ID (1-5, 2-10, ...)
data_storage = {}

for root, dirs, files in os.walk(root_path):
    for filename in files:
        if filename in ['google.tif', 'theos.tif', 'masks.tif']:
            path_parts = root.split(os.sep)
            print(path_parts)
            # หา ID (ตัวเลข-ตัวเลข)
            file_id = "unknown"
            for part in reversed(path_parts):
                if folder_pattern.match(part):
                    file_id = part
                    break
            
            if file_id not in data_storage:
                data_storage[file_id] = {}
            
            # เก็บ Path เต็มแยกตามประเภทภายใต้ ID นั้นๆ
            file_type = filename.split('.')[0] # google, theos, masks
            data_storage[file_id][file_type] = os.path.join(root, filename)

# ตอนนี้ data_storage จะมีหน้าตาประมาณนี้:
# {'1-5': {'google': 'path/to/google.tif', 'masks': 'path/to/masks.tif', ...}, '2-10': {...}}

['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-1']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-1']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-1', 'samgeo2mask']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-10']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-10']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-10', 'samgeo2mask']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-2']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-2']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-2', 'samgeo2mask']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-3']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-3']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-3', 'samgeo2mask']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-4']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-4']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-4', 'samgeo2mask']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-5']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-5']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1-5', 'samgeo2mask']
['E:/GISTDA/geo/GeoAnnotation/raw_data', '1

# Change prefix and save to target folder

In [8]:
import shutil

output_path = r'E:/GISTDA/geo/GeoAnnotation/Result'

if not os.path.exists(output_path):
    os.makedirs(output_path)

for file_id, files_dict in data_storage.items():
    for file_type, original_full_path in files_dict.items():
        
        extension = os.path.splitext(original_full_path)[1]
        
        new_filename = f"{file_id}_{file_type}{extension}"
        
        destination_path = os.path.join(output_path, new_filename)
        
        try:
            # ใช้ copy2 เพื่อรักษา metadata ของไฟล์เดิมไว้
            shutil.copy2(original_full_path, destination_path)
            print(f"Copied: {new_filename}")
        except Exception as e:
            print(f"Error copying {file_id}: {e}")

print("--- การจัดการไฟล์เสร็จสิ้น ---")

Copied: 1-1_google.tif
Copied: 1-1_theos.tif
Copied: 1-1_masks.tif
Copied: 1-10_google.tif
Copied: 1-10_theos.tif
Copied: 1-10_masks.tif
Copied: 1-2_google.tif
Copied: 1-2_theos.tif
Copied: 1-2_masks.tif
Copied: 1-3_google.tif
Copied: 1-3_theos.tif
Copied: 1-3_masks.tif
Copied: 1-4_google.tif
Copied: 1-4_theos.tif
Copied: 1-4_masks.tif
Copied: 1-5_google.tif
Copied: 1-5_theos.tif
Copied: 1-5_masks.tif
Copied: 1-6_google.tif
Copied: 1-6_theos.tif
Copied: 1-6_masks.tif
Copied: 1-7_google.tif
Copied: 1-7_theos.tif
Copied: 1-7_masks.tif
Copied: 1-8_google.tif
Copied: 1-8_theos.tif
Copied: 1-8_masks.tif
Copied: 1-9_google.tif
Copied: 1-9_theos.tif
Copied: 1-9_masks.tif
Copied: 10-1_google.tif
Copied: 10-1_theos.tif
Copied: 10-1_masks.tif
Copied: 10-10_google.tif
Copied: 10-10_theos.tif
Copied: 10-10_masks.tif
Copied: 10-2_google.tif
Copied: 10-2_theos.tif
Copied: 10-2_masks.tif
Copied: 10-3_google.tif
Copied: 10-3_theos.tif
Copied: 10-3_masks.tif
Copied: 10-4_google.tif
Copied: 10-4_theos.t

In [9]:
import os
import re

# กำหนด Path ไปยังโฟลเดอร์ Sorted_Data
mask_root = r'E:\GISTDA\geo\GeoAnnotation\Sorted_Data'

# สร้าง Dictionary ใหม่ (หรือ Reset ของเดิม)
data_storage = {}

# Regex สำหรับแยก "ID" และ "ประเภทไฟล์" 
# รองรับรูปแบบ: 1-1_masks.tif, 10-10_google.tif
file_pattern = re.compile(r'^(\d+-\d+)_(\w+)\.tiff?$')

print(f"Reading files from: {mask_root}")

# วนลูปอ่านไฟล์ทั้งหมดในโฟลเดอร์
for filename in os.listdir(mask_root):
    match = file_pattern.match(filename)
    if match:
        file_id = match.group(1)   # เช่น '1-1'
        file_type = match.group(2) # เช่น 'masks', 'google', 'theos'
        
        # ตรวจสอบและสร้างโครงสร้าง dict สำหรับ file_id นั้นๆ
        if file_id not in data_storage:
            data_storage[file_id] = {}
            
        # เก็บ Path เต็มลงใน Dictionary
        data_storage[file_id][file_type] = os.path.join(mask_root, filename)

# --- แสดงผลสรุป ---
print(f"Total IDs found: {len(data_storage)}")
# ลองปริ้นตัวอย่างตัวแรกออกมาดู
if data_storage:
    sample_id = list(data_storage.keys())[5]
    print(f"Sample Entry ({sample_id}): {data_storage[sample_id]}")

Reading files from: E:\GISTDA\geo\GeoAnnotation\Sorted_Data
Total IDs found: 100
Sample Entry (1-5): {'google': 'E:\\GISTDA\\geo\\GeoAnnotation\\Sorted_Data\\1-5_google.tif', 'masks': 'E:\\GISTDA\\geo\\GeoAnnotation\\Sorted_Data\\1-5_masks.tif', 'theos': 'E:\\GISTDA\\geo\\GeoAnnotation\\Sorted_Data\\1-5_theos.tif'}


# Data Partition (16 grid)

In [11]:
import os
import re
import cv2 
from utils.tools import setup_polygon, save_stats, read_npz
import leafmap.leafmap as leafmap 


root_path = r'E:\GISTDA\geo\GeoAnnotation\Total'
folder_pattern = re.compile(r'\d+-\d+') 
data_storage = {}

for root, dirs, files in os.walk(root_path):
    for filename in files:
        # ตรวจสอบไฟล์เป้าหมาย
        if filename in ['google.tif', 'theos.tif', 'masks.tif']:
            # หา ID (เช่น 1-1) จาก Path
            match = folder_pattern.search(root)
            if match:
                file_id = match.group()
                if file_id not in data_storage:
                    data_storage[file_id] = {}
                
                # เก็บ Path เต็ม โดยใช้ชื่อไฟล์เป็น Key (ตามที่คุณต้องการ re-check)
                data_storage[file_id][filename] = os.path.join(root, filename)


crs_source = "EPSG:4326"
crs_target = "EPSG:32647"
output_base_dir = r'E:\GISTDA\geo\GeoAnnotation\raw_data\Result'
os.makedirs(output_base_dir, exist_ok=True)


for file_id, files in data_storage.items():
    # ดึง Path โดยใช้ Key ที่มีนามสกุล .tif
    google_path = files.get('google.tif')
    theos_path  = files.get('theos.tif')
    mask_path   = files.get('masks.tif')

    # ต้องมีอย่างน้อย Google และ Mask ถึงจะเริ่มตัด
    if google_path and mask_path:
        # กำหนดไฟล์ต้นทางสำหรับตัด (ลำดับความสำคัญ: warped > ต้นฉบับ)
        def get_warped_or_raw(original_path, warped_name):
            if not original_path: return None
            dir_name = os.path.dirname(original_path)
            warped_path = os.path.join(dir_name, warped_name)
            return warped_path if os.path.exists(warped_path) else original_path

        src_google = get_warped_or_raw(google_path, "warped_google.tif")
        src_theos  = get_warped_or_raw(theos_path, "warped_theos.tif")
        src_mask   = mask_path

        # อ่าน Stats จากโฟลเดอร์ Google
        stats_path = os.path.join(os.path.dirname(google_path), "stats.npz")
        
        if os.path.exists(stats_path):
            stats_ = read_npz(stats_path)
            lat_s, lon_s = stats_["lat_start_temp"], stats_["long_start_temp"]
            lat_e, lon_e = stats_["lat_end_temp"], stats_["long_end_temp"]

            n_splits = 4
            lat_step = (lat_e - lat_s) / n_splits
            long_step = (lon_e - lon_s) / n_splits

            file_num = 1
            for i in range(n_splits):
                for j in range(n_splits):
                    
                    curr_s_lat = lat_s + (i * lat_step)
                    curr_e_lat = lat_s + ((i + 1) * lat_step)
                    curr_s_lon = lon_s + (j * long_step)
                    curr_e_lon = lon_s + ((j + 1) * long_step)
                    
                    poly, _, _ = setup_polygon(curr_s_lon, curr_s_lat, curr_e_lon, curr_e_lat, crs_source, crs_target)

                    # --- Export ไฟล์ทั้ง 3 ประเภท (ถ้ามี) ---
                    configs = [
                        (src_google, "google.tif"),
                        (src_theos,  "theos.tif"),
                        (src_mask,   "masks.tif")
                    ]

                    for src_file, suffix in configs:
                        if src_file and os.path.exists(src_file):
                            out_name = f"{file_id}-{file_num}_{suffix}"
                            leafmap.clip_image(src_file, poly, os.path.join(output_base_dir, out_name))

                    file_num += 1
            #print(f"Done: {file_id} (16 sub-images)")

Reading input: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-1_google.tif

Updating dataset tags...
Writing output to: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-1_google.tif
Reading input: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-1_theos.tif

Updating dataset tags...
Writing output to: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-1_theos.tif
Reading input: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-1_masks.tif

Updating dataset tags...
Writing output to: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-1_masks.tif
Reading input: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-2_google.tif

Updating dataset tags...
Writing output to: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-2_google.tif
Reading input: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-2_theos.tif

Updating dataset tags...
Writing output to: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-2_theos.tif
Reading input: E:\GISTDA\geo\GeoAnnotation\raw_data\Result\1-2-2_masks.tif

Updating dataset tag

# Recheck the Result

In [ ]:
import os
import re
from collections import defaultdict

result_path = r'E:\GISTDA\geo\GeoAnnotation\Result'

file_pattern = re.compile(r'^(\d+-\d+)-(\d+)_(\w+)\.tif$')

audit_data = defaultdict(lambda: defaultdict(list))

print(f"Checking files in: {result_path} ---")


files_in_dir = os.listdir(result_path)
for filename in files_in_dir:
    match = file_pattern.match(filename)
    if match:
        area_id = match.group(1)
        file_num = int(match.group(2))
        file_type = match.group(3) # google, theos, หรือ masks
        audit_data[area_id][file_num].append(file_type)

missing_report = []


for area_id in sorted(audit_data.keys(), key=lambda x: [int(i) for i in x.split('-')]):
    # เช็คว่าลำดับ 1-16 ครบไหม
    for num in range(1, 17):
        existing_types = audit_data[area_id][num]
        
        
        required_types = {'google', 'theos', 'masks'}
        missing_from_num = required_types - set(existing_types)
        
        if not existing_types:
            missing_report.append(f"Area {area_id}: ลำดับที่ {num} หายไปทั้งชุด")
        elif missing_from_num:
            missing_report.append(f"Area {area_id}: ลำดับที่ {num} ขาดไฟล์ {list(missing_from_num)}")


if not missing_report:
    print("Success (1-16)")
else:
    print(f"Error {len(missing_report)} จุด:")
    for report in missing_report:
        print(report)


print(f"\nFound {len(audit_data)} Area IDs")

In [ ]:
import os
import re
from collections import defaultdict

result_path = r'E:\GISTDA\geo\GeoAnnotation\Result'

file_pattern = re.compile(r'^(\d+-\d+)-\d+_\w+\.tif$')

# เก็บข้อมูล
stats = defaultdict(int)

if not os.path.exists(result_path):
    print(f"Folder not Found: {result_path}")
else:
    files = os.listdir(result_path)
    for filename in files:
        match = file_pattern.match(filename)
        if match:
            area_id = match.group(1)
            stats[area_id] += 1

    # --- แสดงผลสรุป ---
    print(f"{'Area ID':<15} | {'Total Files':<12} | {'Status'}")
    print("-" * 45)

    # เรียงลำดับ ID ตามตัวเลข
    sorted_ids = sorted(stats.keys(), key=lambda x: [int(i) for i in x.split('-')])
    
    for aid in sorted_ids:
        count = stats[aid]
        # ค่าที่ควรจะเป็นคือ 16 grids * 3 types (google, theos, masks) = 48
        status = "OK" if count == 48 else f"⚠️ Incomplete ({count}/48)"
        print(f"{aid:<15} | {count:<12} | {status}")

    print("-" * 45)
    print(f"Total: {len(stats)} Area IDs | net: {len(files)} files")